In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Load the data we saved in the previous notebook
df = pd.read_csv("../transaction_data.csv")
print("Data Shape:", df.shape)
display(df.head())

Data Shape: (6362620, 15)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_encoded,is_oldbalanceOrg_zero,is_newbalanceOrig_zero,is_oldbalanceDest_zero,is_newbalanceDest_zero,log_amount
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,3,0,0,1,1,9.194276
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,3,0,0,1,1,7.531166
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0,4,0,1,1,1,5.204007
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0,1,0,1,0,1,5.204007
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,3,0,0,1,1,9.364703


In [4]:
# Logic: Fraud transactions often have mismatched balances.

# Origin Account Balance Errors
df["balance_change_orig"] = df["newbalanceOrig"] - df["oldbalanceOrg"]
df["expected_balance_change_orig"] = df["oldbalanceOrg"] - df["amount"]
df["balance_error_orig"] = df["newbalanceOrig"] - df["expected_balance_change_orig"]

# Destination Account Balance Errors
df["balance_change_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["expected_balance_change_dest"] = df["oldbalanceDest"] + df["amount"]
df["balance_error_dest"] = df["newbalanceDest"] - df["expected_balance_change_dest"]

print("Balance Error Features Created.")

Balance Error Features Created.


In [5]:
# Amount-to-Balance Ratios
# Ratio of transaction amount to the existing balance (High ratio = Suspicious)
# Using np.where to avoid division by zero error

df["amount_to_oldbalance_orig_ratio"] = np.where(
    df["oldbalanceOrg"] > 0,
    df["amount"] / df["oldbalanceOrg"],
    0
)

df["amount_to_oldbalance_dest_ratio"] = np.where(
    df["oldbalanceDest"] > 0,
    df["amount"] / df["oldbalanceDest"],
    0
)

print("Amount-to-Balance Ratio Features Created.")

Amount-to-Balance Ratio Features Created.


In [6]:
# Did the transaction completely empty the sender's account?
df["sender_account_emptied"] = (
    (df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)
).astype(int)

# Did the receiver get exactly the amount from zero balance?
df["dest_received_large_amount"] = (
    (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] > 0)
).astype(int)

print("Account Emptying Indicators Created.")

Account Emptying Indicators Created.


In [7]:
# Flagging transactions that fall in the top 5% of all transaction amounts
large_threshold = df["amount"].quantile(0.95)
df["is_large_transaction"] = (df["amount"] > large_threshold).astype(int)

print(f"Large Transaction Threshold set at: {large_threshold:.2f}")

Large Transaction Threshold set at: 518634.20


In [ ]:
# Dividing the 'step' into 6 time buckets (like different times of the day/week)
max_step = df["step"].max()
bins = np.linspace(0, max_step, 7)
df["step_bucket"] = pd.cut(df["step"], bins=bins, labels=False, include_lowest=True)

print("Time-based Step Buckets Created.")